# 00. Диагностика без обучения

Этот ноутбук ничего не обучает. Он отвечает на четыре вопроса, от которых зависит весь план:

1. Совмещены ли пары LQ–HQ с субпиксельной точностью (иначе потери с фазой учат шуму)?
2. Сколько высоких частот теряет сам VAE (потолок для любого дообучения LoRA)?
3. Что происходит с точностью при росте `lambda_sem` (можно ли «накрутить» VoL/GT без пользы)?
4. Чего модели не хватает больше — амплитуды или фазы?

Запустите его первым. На T4 занимает около получаса (оценка, не замер).

**Про выборки.** Решения принимаются по val. Test используется только в пунктах 1 и 2: это описательные проверки
данных и VAE, они не зависят от выбора модели, а числа пригодятся в дипломе. Перебор `lambda_sem` и тест подмены
(пункты 3 и 4) идут только на val. На test модель с зафиксированными настройками прогоняется один раз, в итоговой оценке.

## Инструкция

1. **Настройки ноутбука (панель справа).** Accelerator — `GPU T4 x2`, Internet — `On`, Persistence — `Files only`.
2. **Подключите данные (Add Input):**
   - обучающие HQ и валидационные HQ (LQ создаются в ноутбуке ячейкой `make_lq`);
   - датасет с Stable Diffusion 2.1-base (папки `unet`, `vae`, `tokenizer`, `text_encoder`) и релизный чекпоинт `pisa_sr.pkl`;
   - код берётся из GitHub (`git clone` в первой ячейке), поэтому Internet — `On`.
3. **Ячейки подготовки.** Первые ячейки клонируют код, собирают папку SD 2.1-base (в датасете нет `scheduler`, конфиг
   создаётся автоматически) и создают LQ. В ячейке «Пути» впишите свои пути; ячейка «Проверка» сразу скажет, чего не хватает.
4. **Запускайте ячейки по порядку.** Обучение печатает строку каждые 20 шагов и валидацию каждые `EVAL_EVERY` шагов (по умолчанию 500).
5. **Время.** Сессия Kaggle ограничена; если этап не успевает, уменьшите `STEPS` или запускайте этапы по отдельности
   (см. раздел «Если сессия оборвалась» в конце).
6. **Сохраните результат.** Последняя ячейка упаковывает логи и чекпоинты в архив. Нажмите `Save Version` →
   `Save & Run All` либо скачайте архив из `/kaggle/working`. Для сравнения экспериментов
   (ноутбук `08_compare_all`) подключите сохранённые результаты как входные данные.

**Что смотреть в результатах**

- `curve.png` и `curve_*.csv` — LPIPS, VoL/GT и ошибка в высокочастотных полосах по шагам;
- `hf_err` — главная метрика: 1 = детали не восстановлены, больше 1 = выдуманы, меньше — лучше;
- `eff_gsd_cm` — эффективное разрешение в сантиметрах (меньше — лучше);
- `train_log_*.csv` — значение, вес и норма градиента каждого члена потерь. Если норма градиента частотного
  члена меньше 5 % от главного (L2 в pix-этапе, LPIPS в sem-этапе), член почти не влияет — увеличьте вес.

In [ ]:
# Код из GitHub. Сначала уходим из папки репозитория, иначе rm -rf удалит текущий каталог ядра
%cd /kaggle/working
!rm -rf /kaggle/working/PiSA-SR
!git clone https://github.com/veronika14may/PiSA-SR
%cd /kaggle/working/PiSA-SR

In [ ]:
!mkdir -p /kaggle/working/sd_v2_1_full
!SRC=/kaggle/input/datasets/vende14/sd-v2-1/sd_v2_1; \
for d in unet vae text_encoder; do ln -sfn $SRC/$d /kaggle/working/sd_v2_1_full/$d; done; \
ls -la /kaggle/working/sd_v2_1_full

# В датасете sd_v2_1 нет папки scheduler — создаём конфиг SD 2.1-base вручную
import os, json, subprocess, sys
subprocess.run([
    sys.executable, "-m", "freq_exp.prepare_sd_tokenizer",
    "--sd-dir", "/kaggle/working/sd_v2_1_full",
    "--source", "/kaggle/input/datasets/vende14/sd-v2-1/sd_v2_1/tokenizer",
], check=True)
sched = "/kaggle/working/sd_v2_1_full/scheduler"
if os.path.islink(sched):      # остался битый симлинк с прошлого запуска
    os.remove(sched)
os.makedirs(sched, exist_ok=True)
json.dump({
    "_class_name": "PNDMScheduler", "beta_end": 0.012, "beta_schedule": "scaled_linear",
    "beta_start": 0.00085, "clip_sample": False, "num_train_timesteps": 1000,
    "prediction_type": "epsilon", "set_alpha_to_one": False, "skip_prk_steps": True,
    "steps_offset": 1, "trained_betas": None,
}, open(sched + "/scheduler_config.json", "w"), indent=2)

In [ ]:
NOTEBOOK_TAG = "00_diagnostics"

# ==== Пути: отредактируйте под свои датасеты ====
CODE_SRC  = "/kaggle/working/PiSA-SR"                       # клон репозитория (внутри должна быть freq_exp)
SD_PATH   = "/kaggle/working/sd_v2_1_full"                  # Stable Diffusion 2.1-base
INIT_CKPT = "/kaggle/input/datasets/vende14/pisa-sr/pisa_sr.pkl"   # релизный чекпоинт PiSA-SR
RAM_PATH  = ""      # файл ram_swin_large_14m.pth; пусто = без RAM (фиксированный промпт)
TRAIN_HQ  = "/kaggle/input/datasets/vende14/data-from-scratch-netherlands/train/hq"
VAL_HQ    = "/kaggle/input/datasets/vende14/hq-lq-netherlands-val/hq_images"
LQ_PRESET = "diffbir"   # деградация LQ: "diffbir" — Real-ESRGAN/DiffBIR (как раньше); "mild" — мягкая (вариант B)
_sfx = "" if LQ_PRESET == "diffbir" else "_" + LQ_PRESET      # у каждого пресета своя папка, файлы не смешиваются
TRAIN_LQ  = f"/kaggle/working/lq_train{_sfx}"   # создаются ячейкой make_lq из HQ
VAL_LQ    = f"/kaggle/working/lq_val{_sfx}"

# ==== Параметры обучения ====
OUT_ROOT = "/kaggle/working/experiments/freq"   # сюда пишутся логи и чекпоинты
STEPS    = 4000    # шагов на этап (как в ваших прежних запусках)
EVAL_EVERY = 500   # валидация каждые столько шагов; чаще — точнее кривая, но дольше обучение
ACCUM    = 1       # накопление градиента: 4 сглаживает кривые, но за те же шаги даёт вчетверо меньше обновлений
EMA      = 0       # 0.999 включает EMA весов LoRA (сохраняется отдельный *_ema.pkl)
FDL_CROP = 256     # окно для FDL: 256 экономит память; 0 = весь кроп 512
AUTO     = 0.3     # калибровка весов частотных членов по норме градиента; 0 = взять веса как есть

# ==== Новая нарезка (ноутбук 10_make_splits): впишите имя датасета, куда сохранили dataset/{train,val,test}/hq ====
DATA_ROOT = "/kaggle/input/datasets/vende14/<имя-датасета>"
TRAIN_HQ = f"{DATA_ROOT}/train/hq"
VAL_HQ   = f"{DATA_ROOT}/val/hq"      # по val принимаются решения (вопросы 3 и 4)
TEST_HQ  = f"{DATA_ROOT}/test/hq"     # test — только описательные проверки 1 и 2; решения по нему не принимать
TEST_LQ  = f"/kaggle/working/lq_test{_sfx}"

# Чекпоинт для вопросов 3 и 4: релизный или ваш дообученный (например, pisasr_final.pkl из прежних запусков)
CKPT_TO_CHECK = INIT_CKPT
N_DIAG = 0     # 0 = все снимки выборки (val 600, test 1000); снимки берутся равномерно по списку, поэтому охвачены все тайлы и оба класса

In [ ]:
# LQ из HQ без сдвига (повторный запуск пропускает готовые файлы)
!python /kaggle/working/PiSA-SR/freq_exp/make_lq.py --hq_dir {TRAIN_HQ} --out_dir {TRAIN_LQ} --preset {LQ_PRESET}
!python /kaggle/working/PiSA-SR/freq_exp/make_lq.py --hq_dir {VAL_HQ} --out_dir {VAL_LQ} --seed 1 --preset {LQ_PRESET}

In [ ]:
# LQ для test: другой сид (2), чем у val (1), чтобы деградация отличалась
!python /kaggle/working/PiSA-SR/freq_exp/make_lq.py --hq_dir {TEST_HQ} --out_dir {TEST_LQ} --seed 2 --preset {LQ_PRESET}

In [ ]:
import os, subprocess, sys

# --- Проверка ---
!nvidia-smi --query-gpu=name,memory.total --format=csv
missing = [p for p in [CODE_SRC, SD_PATH, INIT_CKPT, TRAIN_LQ, TRAIN_HQ, VAL_LQ, VAL_HQ] if not os.path.exists(p)]
assert not missing, f"Не найдены пути: {missing}"
assert os.path.isdir(os.path.join(CODE_SRC, "freq_exp")), "В клоне нет папки freq_exp"
if RAM_PATH and not os.path.exists(RAM_PATH):
    raise FileNotFoundError(RAM_PATH)
os.chdir(CODE_SRC)

# --- Пакеты: две функции потерь из статей + метрики ---
!pip install -q focal-frequency-loss fdl-pytorch lpips pyiqa
# peft требует torchao>=0.16, а на Kaggle стоит 0.10 — он не нужен
!pip uninstall -y -q torchao
# Если окружение чистое, раскомментируйте установку зависимостей PiSA-SR:
# !pip install -q -r requirements.txt

os.environ.update(SD=SD_PATH, INIT=INIT_CKPT, RAM=RAM_PATH, TRAIN_LQ=TRAIN_LQ, TRAIN_HQ=TRAIN_HQ,
                  VAL_LQ=VAL_LQ, VAL_HQ=VAL_HQ, OUT=OUT_ROOT, STEPS=str(STEPS), ACCUM=str(ACCUM),
                  EMA=str(EMA), FDL_CROP=str(FDL_CROP), AUTO=str(AUTO), EVAL_EVERY=str(EVAL_EVERY))
print("Рабочая папка:", os.getcwd())
!bash freq_exp/experiments.sh list

In [ ]:
assert "<" not in DATA_ROOT, "впишите имя датасета в DATA_ROOT (ячейка «Пути»)"
assert os.path.isdir(TEST_HQ) and os.path.isdir(TEST_LQ), f"нет test: {TEST_HQ}, {TEST_LQ}"

## 1. Совмещение пар LQ–HQ

Средний сдвиг должен быть меньше 0,25 пикселя HQ.

In [ ]:
!python freq_exp/diag_alignment.py --lq_dir {TRAIN_LQ} --hq_dir {TRAIN_HQ} --n 200
!python freq_exp/diag_alignment.py --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --n 200
!python freq_exp/diag_alignment.py --lq_dir {TEST_LQ} --hq_dir {TEST_HQ} --n 200

## 2. Потолок VAE

Эталон кодируется и декодируется без UNet. Если ошибка в ВЧ-полосах близка к 1, узкое место — VAE, и потери на UNet дадут ограниченный эффект. Считается на val и на test.

In [ ]:
!python freq_exp/diag_vae_ceiling.py --pretrained_model_path {SD_PATH} --hq_dir {VAL_HQ} --n {N_DIAG}
!python freq_exp/diag_vae_ceiling.py --pretrained_model_path {SD_PATH} --hq_dir {TEST_HQ} --n {N_DIAG}

## 3. Перебор `lambda_sem`

Если с ростом `lambda_sem` отношение `vol_ratio` идёт к 1, а `hf_err` растёт выше 1, резкость добавляется выдуманными деталями. Только val.

In [ ]:
!python freq_exp/infer_eval.py --pretrained_model_path {SD_PATH} --ckpt {CKPT_TO_CHECK} \
    --lq_dir {VAL_LQ} --hq_dir {VAL_HQ} --out_dir /kaggle/working/sr/diag{_sfx} --n {N_DIAG} \
    --lambda_sem 0 0.5 1.0 1.2 1.5 2.0

## 4. Амплитуда или фаза

Тест подмены из TFDSR для режима `lambda_sem = 1`.

In [ ]:
!python freq_exp/diag_swap_test.py --sr_dir /kaggle/working/sr/diag{_sfx}/lsem_1.0 --hq_dir {VAL_HQ}

## Что делать с результатами

- Сдвиг пар больше 0,25 пикселя — сначала пересоздать LQ, иначе эксперименты с FFL и Fourier Space Loss бессмысленны.
- Потолок VAE низкий — в дипломе это нужно показать отдельной строкой таблицы; ожидания от потерь снизить.
- Основная доля ВЧ-ошибки от фазы — приоритет у ноутбука `03_V1`; от амплитуды — у `02_K1`.